In [1]:
from pathlib import Path
import matplotlib.pyplot as plt
from xdqc.graph import NetworkGraph
from xdqc.io.qasm import load_qasm_program
from xdqc.partition import Partitioner
from xdqc.builder.circuit_extractor import extract_distributed_circuit
from xdqc.verify import verify_distributed_circuit
import openqasm3

In [2]:
sample_circuits = ["simple3.qasm", "qft_n4.qasm"]
sample_networks = ["3comp_1comm_x2.json", "simple1.json"]

In [3]:
repo_root = Path.cwd()
circuit_path = Path(repo_root) / "circuits" / sample_circuits[1]
sample_network_path = Path(repo_root) / "networks" / sample_networks[1]

In [4]:
qasm_program = load_qasm_program(str(circuit_path))
network_graph = NetworkGraph(str(sample_network_path))

In [5]:
partitioner = Partitioner(network_graph, qasm_program)

In [6]:
partitioner.run()

In [7]:
cost = partitioner.cost
schedule = partitioner.schedule

print("cost", cost)
print("schedule", schedule)

cost 4.0
schedule [{QPU(id=0): {1, 2}, QPU(id=1): {0, 3}}, {QPU(id=0): {1, 2}, QPU(id=1): {0, 3}}, {QPU(id=0): {1, 2}, QPU(id=1): {0, 3}}]


In [8]:
# TODO: cleanup path handling

out_path = repo_root / f"{circuit_path.stem}_distributed.qasm"
distributed_prog = extract_distributed_circuit(partitioner)
qasm_str = openqasm3.dumps(distributed_prog)
print(qasm_str)
with out_path.open("w", encoding="utf-8") as f:
  openqasm3.dump(distributed_prog, f)

OPENQASM 3.0;
include "builder/distgates.inc";
include "stdgates.inc";
qubit[2] q0;
qubit[2] q1;
bit[4] c;
x q1[0];
x q0[1];
h q1[0];
rcp(pi / 2) q0[0], q1[0];
h q0[0];
rcp(pi / 4) q0[1], q1[0];
cp(pi / 2) q0[1], q0[0];
h q0[1];
cp(pi / 8) q1[1], q1[0];
rcp(pi / 4) q1[1], q0[0];
rcp(pi / 2) q1[1], q0[1];
h q1[1];
c[0] = measure q1[0];
c[1] = measure q0[0];
c[2] = measure q0[1];
c[3] = measure q1[1];



In [9]:
verify_distributed_circuit(str(circuit_path), out_path)

Verification succeeded: fidelity 0.9992688080029873 is above threshold 0.995


True